# prepare ugr16

fetches the six test weeks of UGR'16 one at a time from the Hugging Face mirror of the official archives, checks each against the size and sha256 of the mirror's manifest, keeps every k-th flow of the week in file order and writes the processed version named in the data config into the project folder on Drive. an archive lives on the local disk of the session only while its week is read, and a week already written is found again when the session restarts. a processed version is written once and never overwritten, so a change to the processing gets a new version in the config.

the GitHub repository of this project and its access token are read from Colab Secrets, GITHUB_REPO and GITHUB_TOKEN, so nothing personal is stored in this notebook.

In [ ]:
CONFIG = "configs/data/ugr16.yaml"
BRANCH = "main"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
from google.colab import userdata

CODE_DIR = "/content/forge-nids"
repo = userdata.get("GITHUB_REPO")
try:
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = ""
auth = f"x-access-token:{token}@" if token else ""
url = f"https://{auth}github.com/{repo}.git"


def git(*args, cwd=None):
    out = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stderr.replace(token, "***") if token else out.stderr)
    return out.stdout.strip()


if os.path.isdir(CODE_DIR):
    # the Colab copy always matches GitHub exactly, local edits here are discarded
    git("fetch", "--quiet", url, BRANCH, cwd=CODE_DIR)
    git("checkout", "--quiet", "-B", BRANCH, "FETCH_HEAD", cwd=CODE_DIR)
else:
    git("clone", "--quiet", "--branch", BRANCH, url, CODE_DIR)
    git("remote", "set-url", "origin", f"https://github.com/{repo}.git", cwd=CODE_DIR)
print("code at", git("log", "-1", "--format=%h %s", cwd=CODE_DIR))
if not os.path.isfile(f"{CODE_DIR}/{CONFIG}"):
    tracked = git("ls-files", "configs", cwd=CODE_DIR) or "nothing"
    raise FileNotFoundError(f"{CONFIG} is not in branch {BRANCH} on GitHub, configs tracked there are {tracked}")

In [ ]:
%pip install --quiet -e /content/forge-nids

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/forge-nids")
os.environ["FORGE_STORAGE"] = str(DRIVE_ROOT)
for sub in ["notebooks", "data/raw", "data/processed", "runs", "exports"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

In [ ]:
import sys

cmd = [sys.executable, "scripts/prepare.py", CONFIG]
# output is streamed line by line, a child process would otherwise print nowhere visible in Colab
proc = subprocess.Popen(cmd, cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    raise RuntimeError("the preparation failed, the lines above show where")

In [ ]:
import yaml
from IPython.display import Markdown, display

dataset = yaml.safe_load(open(f"{CODE_DIR}/{CONFIG}"))["dataset"]
folder = DRIVE_ROOT / "data" / "processed" / dataset["name"] / dataset["processed_version"]
print(folder)
display(Markdown((folder / "readme.md").read_text()))